# Practical 13 --- Watching a Model Get Worse

**Drift detection, alert rules, and an audit of who the model serves badly**

SCSE3040 Machine Learning Operations &middot; Bennett University &middot; Session 2026-27

| | |
|---|---|
| **Lectures this follows** | L26, L28 |
| **Course Outcome** | CO5 |
| **Lab duration** | 120 minutes |
| **Memory needed** | about 500 MB (fine on a 4 GB or 8 GB machine) |
| **Extra software** | nothing beyond the course venv |
| **Marks** | 10 |

---

## Aim

1. Measure a deployed model's error week by week instead of once.
2. Detect when the incoming data stops looking like the training data.
3. Write an alert rule that fires on a real problem, not on noise.
4. Audit whether the model serves some groups of customers worse.

---

## What you need to know first


This is the second-last practical, and it covers the thing most courses leave
out. Your model is deployed. Nobody is watching it. It is quietly getting worse.
The last practical, P14, repairs what you find today.

It gets worse because the world moves. A new restaurant partner has slower
kitchens. The monsoon arrives. The company expands into a suburb 15 km out,
when every delivery you trained on was under 12 km. The code did not change; the
world did. That is **drift**.

There are two kinds and you must be able to tell them apart.

**Data drift** means the inputs changed shape --- distances are longer than they
used to be. You can detect this **immediately**, because you see every request
as it arrives.

**Concept drift** means the relationship changed --- the same 5 km trip now
takes longer than it used to, because traffic rules changed. To detect this you
need the **true answer**, which arrives only when the delivery is complete. In a
delivery service that is an hour later. In credit scoring it can be a year.

That gap is why we watch the inputs as well as the errors. The standard measure
is **PSI**, the Population Stability Index: it compares the shape of a feature
now against the shape it had in training. The conventional reading is below 0.1
stable, 0.1 to 0.25 worth watching, above 0.25 a real shift.

Then there is the part that is not statistics. A model with a good average can
still be bad for particular people. Yours might predict well for short trips in
dry weather and badly for long trips in the rain --- and the customers who
suffer are always the ones on the edge of town. An **average hides a
distribution**. So today you finish by splitting the error by group and looking
for who the model serves worst.

That last check is the whole of L28 in one habit: not "is my model accurate?"
but **"accurate for whom?"**

---

## Before you start

- Practicals P02 and P07 are finished. You can train a model and serve it.
- You remember MAE from P02: the average miss, in minutes.

### How to run a cell

Click on a grey code cell, then press **Shift + Enter**. The cell runs and
the cursor moves to the next one. A number appears in the `[ ]` on the left
when the cell has finished.

**Run the cells in order, from the top.** A later cell almost always uses
something an earlier cell created. If you jump ahead you will see a
`NameError`, which just means "you have not made that thing yet".

If everything goes wrong, use the menu: **Kernel -> Restart Kernel and Clear
All Outputs**, then start again from the first cell. Nothing is damaged by
doing this.

In [ ]:
# Step 0 --- check the workbench before we start.
# This cell only looks; it changes nothing. Run it and read the last line.

import sys
from pathlib import Path

print("Python  :", sys.version.split()[0])
print("Folder  :", Path.cwd().name)

_missing = []
for _name in ['numpy', 'pandas', 'sklearn', 'matplotlib']:
    try:
        __import__(_name)
    except ImportError:
        _missing.append(_name)

for _name in ['numpy', 'pandas', 'sklearn', 'matplotlib']:
    _mark = "missing" if _name in _missing else "ok"
    print(f"  {_name:<14} {_mark}")

if _missing:
    print()
    print("STOP. Some libraries are missing:", ", ".join(_missing))
    print("Ask your instructor to run the setup in labs/SETUP.md.")
else:
    print()
    print("All good. You can carry on to Step 1.")

---

## Step 0b --- the dataset

Every practical in this course uses the same 600 food deliveries.
The next cell makes sure the file is there.

In [ ]:
# The delivery-time dataset every practical in this course uses.
# If the file is missing we build it again from the same seed, so every
# student in the room gets byte-for-byte the same 600 rows.

import csv
from pathlib import Path

import numpy as np

SEED = 42
N_ROWS = 600
DATA = Path("..") / "data" / "delivery_times.csv"


def make_delivery_csv(path=DATA):
    """Write the 600-row delivery dataset. Same formula as the lectures."""
    rng = np.random.default_rng(SEED)
    distance_km = np.round(rng.uniform(0.5, 12.0, N_ROWS), 2)
    prep_time_min = np.round(rng.uniform(5, 30, N_ROWS), 0)
    traffic_level = rng.integers(1, 4, N_ROWS)
    rain = rng.binomial(1, 0.25, N_ROWS)
    delivery_min = np.round(
        6.0
        + 3.1 * distance_km
        + 0.65 * prep_time_min
        + 4.2 * traffic_level
        + 5.5 * rain
        + rng.normal(0, 2.5, N_ROWS),
        1,
    )
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(["distance_km", "prep_time_min", "traffic_level",
                    "rain", "delivery_min"])
        for i in range(N_ROWS):
            w.writerow([distance_km[i], int(prep_time_min[i]),
                        int(traffic_level[i]), int(rain[i]), delivery_min[i]])
    return path


if not DATA.exists():
    make_delivery_csv()
    print("dataset rebuilt ->", DATA)
else:
    print("dataset found   ->", DATA)

---

# Walkthrough

Read each step, then run its cell.

### Step 1 --- Train the model that will go into production

This is the model from P02, trained on data up to today. In the real
world you would deploy it and walk away. Today we will watch it.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

WORK = Path("work")
WORK.mkdir(exist_ok=True)
FEATURES = ["distance_km", "prep_time_min", "traffic_level", "rain"]

orders = pd.read_csv(DATA)
X_train, X_test, y_train, y_test = train_test_split(
    orders[FEATURES], orders["delivery_min"],
    test_size=0.2, random_state=42,
)
model = LinearRegression().fit(X_train, y_train)

launch_mae = mean_absolute_error(y_test, model.predict(X_test))
print(f"MAE at launch: {launch_mae:.2f} minutes")
print("Deployed. Nobody is watching.")

### Step 2 --- Eight weeks of real traffic arrive

We simulate what happens next. Weeks 1 to 4 look like the training
data. From week 5 the company expands into a suburb, so trips get
longer.

Two separate things change, and the difference is the whole theory
section made concrete.

**The inputs move.** Distances stretch beyond anything in the
training data. That is **data drift**.

**The rule changes too.** Past about 12 km a rider leaves the city
roads, so every extra kilometre out there costs more minutes than a
kilometre in town. The training data contained no such trips, so the
model never learned this. That is **concept drift**.

Read `long_trip_penalty` in the code below: it is zero for every
trip under 12 km, which is why weeks 1 to 4 look completely normal.

In [ ]:
SEED = 7
rng = np.random.default_rng(SEED)
ORDERS_PER_WEEK = 150


def week_of_orders(week):
    """One week of live traffic. From week 5, trips get longer."""
    stretch = 1.0 if week <= 4 else 1.0 + 0.22 * (week - 4)

    distance_km = np.round(
        rng.uniform(0.5, 12.0, ORDERS_PER_WEEK) * stretch, 2)
    prep_time_min = np.round(
        rng.uniform(5, 30, ORDERS_PER_WEEK), 0).astype(int)
    traffic_level = rng.integers(1, 4, ORDERS_PER_WEEK)
    rain = rng.binomial(1, 0.25, ORDERS_PER_WEEK)

    # Beyond 12 km the rider is on highways, not city roads, and
    # every extra kilometre costs more. No trip in the training
    # data was that long, so the model has never seen this rule.
    long_trip_penalty = 1.8 * np.maximum(distance_km - 12.0, 0.0)

    true_minutes = np.round(
        6.0 + 3.1 * distance_km + 0.65 * prep_time_min
        + 4.2 * traffic_level + 5.5 * rain
        + long_trip_penalty
        + rng.normal(0, 2.5, ORDERS_PER_WEEK), 1)

    return pd.DataFrame({
        "week": week,
        "distance_km": distance_km,
        "prep_time_min": prep_time_min,
        "traffic_level": traffic_level,
        "rain": rain,
        "actual_min": true_minutes,
    })


live = pd.concat([week_of_orders(w) for w in range(1, 9)],
                 ignore_index=True)
live["predicted_min"] = model.predict(live[FEATURES])

print("orders received:", len(live))
print(live.groupby("week")["distance_km"].mean().round(2)
      .to_string())

Look at the average distance per week. Weeks 1 to 4 sit around
6 km; from week 5 it climbs. That is data drift, and it happened
without anybody deploying anything.

### Step 3 --- Measure the error week by week

One MAE for the whole period would hide everything. Measure it per
week, the way a real dashboard would.

In [ ]:
weekly = (live.assign(
              error=lambda d: (d["actual_min"] - d["predicted_min"]).abs())
          .groupby("week")
          .agg(orders=("error", "size"),
               mae=("error", "mean"),
               mean_distance=("distance_km", "mean"))
          .round(2))

print(weekly.to_string())
print()
print(f"MAE at launch was {launch_mae:.2f} minutes.")

Weeks 1 to 4 sit near the launch figure. Then it climbs, and keeps
climbing. Nobody changed the code. Nobody was told.

### Step 4 --- Draw it, because a table hides a trend

The same numbers as a picture. This is what a Grafana dashboard
shows a team every morning.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.6))

left.plot(weekly.index, weekly["mae"], marker="o", color="#9E3B33")
left.axhline(launch_mae, linestyle="--", color="#6B7785",
             label=f"launch MAE {launch_mae:.2f}")
left.axhline(launch_mae * 1.5, linestyle=":", color="#B4703A",
             label="alert threshold")
left.set_title("Prediction error by week")
left.set_xlabel("week")
left.set_ylabel("MAE (minutes)")
left.legend(fontsize=8)

right.plot(weekly.index, weekly["mean_distance"], marker="s",
           color="#2C5F79")
right.set_title("Average distance by week")
right.set_xlabel("week")
right.set_ylabel("km")

plt.tight_layout()
plt.show()

Two lines, same shape. The inputs moved, so the error followed. A
dashboard showing only the left chart tells you *that* something
broke; showing both tells you *what*.

### Step 5 --- Which way is it wrong? Find where the model breaks

MAE throws the sign away. A model that is 5 minutes early and one
that is 5 minutes late get the same MAE, but a customer only
complains about one of them.

So keep the sign. **Signed error** is actual minus predicted:

- **above 0**: the food arrived *later* than the app promised;
- **below 0**: it arrived *earlier*.

The left chart puts every order from all eight weeks on one
picture, against its distance. The dotted line is the longest
trip in the training data.

**Predict first:** if the model were simply a bit worse
everywhere, what shape would the dots make?

In [ ]:
live["signed_error"] = live["actual_min"] - live["predicted_min"]
longest_trained = X_train["distance_km"].max()
seen = live["distance_km"] <= longest_trained

inside, beyond = live[seen], live[~seen]
inside_mae = inside["signed_error"].abs().mean()
beyond_mae = beyond["signed_error"].abs().mean()
beyond_bias = beyond["signed_error"].mean()

print(f"Longest trip in the training data: {longest_trained:.2f} km")
print()
print(f"{'':<24}{'orders':>7}{'MAE':>8}{'avg signed error':>19}")
print(f"{'inside training range':<24}{len(inside):>7}"
      f"{inside_mae:>8.2f}{inside['signed_error'].mean():>+19.2f}")
print(f"{'beyond it':<24}{len(beyond):>7}"
      f"{beyond_mae:>8.2f}{beyond_bias:>+19.2f}")

weekly_bias = live.groupby("week")["signed_error"].mean()

fig, (left, right) = plt.subplots(
    1, 2, figsize=(11, 3.8), gridspec_kw={"width_ratios": [1.6, 1]})

left.scatter(inside["distance_km"], inside["signed_error"], s=8,
             color="#B9BFC7", label="inside training range")
left.scatter(beyond["distance_km"], beyond["signed_error"], s=10,
             color="#9E3B33", label="beyond it")
left.axhline(0, color="#6B7785", linewidth=1)
left.axvline(longest_trained, color="#6B7785", linestyle=":",
             linewidth=1)
left.text(0.02, 0.95, "above 0: later than promised",
          transform=left.transAxes, fontsize=8, va="top",
          color="#6B7785")
left.set_title("Every dot is one order, weeks 1-8", fontsize=10)
left.set_xlabel("distance_km")
left.set_ylabel("actual - predicted (minutes)")
left.legend(fontsize=8, loc="upper center")
left.spines[["top", "right"]].set_visible(False)

right.bar(weekly_bias.index, weekly_bias.values,
          color=["#9E3B33" if b > 1 else "#6B7785"
                 for b in weekly_bias])
right.axhline(0, color="#6B7785", linewidth=1)
right.set_title("Average signed error by week", fontsize=10)
right.set_xlabel("week")
right.set_ylabel("minutes late (+) or early (-)")
right.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

**How to read the left chart.** Walk along it from left to right.

- Up to the dotted line, the grey cloud sits evenly on the zero
  line. Inside the range it was trained on, the model is still as
  good as it was at launch --- compare the MAE in the table with
  the launch figure.
- Past the dotted line, the red dots leave the cloud and climb in
  a straight ramp. Every extra kilometre out there adds minutes
  the model does not know about. That ramp *is* the
  `long_trip_penalty` from Step 2, rediscovered from the data
  alone.
- Almost every red dot is **above** zero. The model is not
  randomly wrong out there; it is wrong in one direction. It
  promises the food too early, every time.

**The right chart** says the same thing week by week: bias near
zero for weeks 1 to 4, then customers start waiting longer than
promised.

So the answer to "predict first": a model that is a bit worse
everywhere makes a *wider* cloud, still centred on zero. What we
see instead is a *bend* at the edge of the training data. A model
is only trustworthy inside the range it was trained on.

### Step 6 --- Detect the drift in the inputs

The error chart needs the true delivery times, which arrive an hour
late. The input chart does not --- you can compute it the moment a
request lands.

**PSI** compares two distributions bucket by bucket. We cut the
training data into ten buckets, then ask what share of the new data
falls into each.

In [ ]:
def psi(reference, current, buckets=10):
    """Population Stability Index between two samples."""
    edges = np.percentile(reference, np.linspace(0, 100, buckets + 1))
    edges[0], edges[-1] = -np.inf, np.inf

    ref_share = np.histogram(reference, bins=edges)[0] / len(reference)
    cur_share = np.histogram(current, bins=edges)[0] / len(current)

    # a tiny floor keeps empty buckets from producing infinity
    ref_share = np.clip(ref_share, 1e-6, None)
    cur_share = np.clip(cur_share, 1e-6, None)

    return float(np.sum((cur_share - ref_share)
                        * np.log(cur_share / ref_share)))


print(f"{'week':<6}{'PSI(distance)':<16}{'reading'}")
print("-" * 48)
for week in range(1, 9):
    current = live.loc[live["week"] == week, "distance_km"]
    score = psi(X_train["distance_km"], current)
    if score < 0.1:
        reading = "stable"
    elif score < 0.25:
        reading = "worth watching"
    else:
        reading = "SHIFTED"
    print(f"{week:<6}{score:<16.3f}{reading}")

Two things to read here.

PSI crosses 0.25 at week 6, while the error was still only starting
to move. Watching the inputs is the **early warning**, because you
do not have to wait for the truth to arrive.

But look at **week 3**: 0.127, in the "worth watching" band, with
absolutely nothing wrong. That is a false alarm from ordinary
randomness. It is why Step 8 does not alert on a single
reading.

### Step 7 --- See the drift: the two shapes PSI compares

PSI is one number about two shapes. Look at the shapes. The grey
histogram is the training data's distances; the blue one is a
single live week. Four weeks are drawn side by side: watch the
blue shape walk off to the right and PSI climb with it.

If the `ipywidgets` library is installed, a **slider** appears
under the panels: drag through weeks 1 to 8 and watch the walk
happen. Without it, the four panels tell the same story.

In [ ]:
edges = np.linspace(0, 24, 25)


def draw_week(week, ax=None):
    """Training distances (grey) against one live week (blue)."""
    own_figure = ax is None
    if own_figure:
        _, ax = plt.subplots(figsize=(6.5, 3.2))
    current = live.loc[live["week"] == week, "distance_km"]
    ax.hist(X_train["distance_km"], bins=edges, density=True,
            color="#B9BFC7", label="training data")
    ax.hist(current, bins=edges, density=True, color="#2C5F79",
            alpha=0.65, label=f"week {week}")
    ax.axvline(12, color="#6B7785", linestyle=":", linewidth=1)
    ax.set_title(f"week {week}: PSI "
                 f"{psi(X_train['distance_km'], current):.3f}",
                 fontsize=10)
    ax.set_xlabel("distance_km")
    ax.legend(fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)
    if own_figure:
        plt.show()          # so the slider redraws in its own box


fig, axes = plt.subplots(1, 4, figsize=(13, 3), sharey=True)
for ax, week in zip(axes, (1, 4, 6, 8)):
    draw_week(week, ax)
axes[0].set_ylabel("share of orders")
plt.tight_layout()
plt.show()

try:
    from ipywidgets import IntSlider, fixed, interact
except ImportError:
    print("(install ipywidgets to get a week slider here)")
else:
    interact(draw_week, ax=fixed(None),
             week=IntSlider(min=1, max=8, step=1, value=1));

Week 1 and the training data are the same shape, and PSI says
so. By week 8 most of the blue is to the right of the dotted
12 km line, where the grey has nothing at all. PSI is simply the
size of that mismatch, bucket by bucket.

### Step 8 --- Write an alert rule that does not cry wolf

A threshold on its own fires on every noisy week and gets muted
within a fortnight. A usable rule needs the problem to **persist**.

Ours: the error must exceed 1.5 times the launch figure for two
weeks running.

In [ ]:
def check_alerts(weekly, launch_mae, multiplier=1.5, in_a_row=2):
    """Fire only when the error stays high for several weeks."""
    threshold = launch_mae * multiplier
    streak, alerts = 0, []

    for week, row in weekly.iterrows():
        if row["mae"] > threshold:
            streak += 1
            if streak >= in_a_row:
                alerts.append({
                    "week": int(week),
                    "mae": float(row["mae"]),
                    "threshold": round(threshold, 2),
                })
        else:
            streak = 0
    return alerts


alerts = check_alerts(weekly, launch_mae)

print(f"threshold: {launch_mae * 1.5:.2f} minutes, "
      f"sustained for 2 weeks")
print()
for alert in alerts:
    print(f"  ALERT week {alert['week']}: MAE {alert['mae']:.2f} "
          f"is above {alert['threshold']}")
if not alerts:
    print("  no alerts")
print()
print("First alert:", alerts[0]["week"] if alerts else "none")

### Step 9 --- Draw the warning and the alarm on one timeline

You now have two signals, and they arrive at different speeds.

- **PSI** (top panel) is ready the moment an order arrives. It is
  the **warning**.
- **MAE** (bottom panel) waits for the true delivery time. Its
  alert rule is the **alarm**.

In the bottom panel a bar is **grey** when the error is normal,
**orange** when it is above the line but the rule is still waiting
to see if it lasts, and **red** when the alert actually fires.

In [ ]:
from matplotlib.patches import Patch

week_psi = pd.Series({
    w: psi(X_train["distance_km"],
           live.loc[live["week"] == w, "distance_km"])
    for w in range(1, 9)})
threshold = launch_mae * 1.5
alert_weeks = {a["week"] for a in alerts}

first_shift = int(week_psi[week_psi >= 0.25].index[0])
first_alert = alerts[0]["week"]

fig, (top, bottom) = plt.subplots(2, 1, figsize=(9, 5.6),
                                  sharex=True)

top.axhspan(0.10, 0.25, color="#B4703A", alpha=0.15)
top.axhspan(0.25, 1.25, color="#9E3B33", alpha=0.10)
for y, word in [(0.04, "stable"), (0.17, "watching"),
                (0.70, "shifted")]:
    top.text(8.45, y, word, fontsize=8, color="#6B7785",
             va="center")
top.plot(week_psi.index, week_psi.values, marker="o",
         color="#2C5F79")
top.set_ylim(0, 1.25)
top.set_ylabel("PSI(distance)")
top.set_title("Inputs: known the moment an order arrives",
              fontsize=10)

colours = []
for week, mae in weekly["mae"].items():
    if week in alert_weeks:
        colours.append("#9E3B33")
    elif mae > threshold:
        colours.append("#B4703A")
    else:
        colours.append("#B9BFC7")
bottom.bar(weekly.index, weekly["mae"], color=colours)
bottom.axhline(threshold, color="#B4703A", linestyle=":")
bottom.set_ylabel("MAE (minutes)")
bottom.set_xlabel("week")
bottom.set_title("Errors: known only once the true times arrive",
                 fontsize=10)
bottom.legend(handles=[
    Patch(color="#B9BFC7", label="normal"),
    Patch(color="#B4703A", label="above the line, waiting"),
    Patch(color="#9E3B33", label="ALERT fired"),
], fontsize=8, loc="upper left")

for ax in (top, bottom):
    ax.axvline(first_shift, color="#2C5F79", linestyle="--",
               linewidth=1, label=f"warning: week {first_shift}")
    ax.axvline(first_alert, color="#9E3B33", linestyle="--",
               linewidth=1, label=f"alarm: week {first_alert}")
    ax.spines[["top", "right"]].set_visible(False)
top.legend(fontsize=8, loc="upper left")
bottom.set_xticks(range(1, 9))
plt.tight_layout()
plt.show()

print(f"Inputs first SHIFTED:    week {first_shift}")
print(f"Error alert first fired: week {first_alert}")
print(f"Warning came {first_alert - first_shift} week(s) "
      f"before the alarm.")

Read the timeline one week at a time.

- **Week 3.** The PSI dot is in the orange band, but the error bar
  underneath is grey. The inputs wobbled; the service did not. Had
  we woken someone up on PSI alone, it would have been for
  nothing.
- **Week 6.** PSI jumps into the red band. The error bar is above
  the line but **orange**: the rule wants a second week before it
  believes it.
- **Week 7.** Second week above the line. The bar turns red and
  the alarm fires.

That gives the standard two-level design real teams use: a PSI
warning opens a ticket ("somebody look at this today"), and only
the error alarm wakes somebody up at night.

### Step 10 --- Triage: what should we do this week?

Every week, a monitoring team asks two yes/no questions:

1. **Did the inputs move?** (PSI at or above 0.25)
2. **Did the error rise?** (MAE above 1.5 times launch)

Two questions give four answers, and each answer means a
different action:

| | error normal | error high |
|---|---|---|
| **inputs stable** | healthy: do nothing | error with no input drift: investigate |
| **inputs moved** | early warning: watch closely | drift confirmed: retrain |

The code puts each of our eight weeks into one of those four boxes,
as a table and as a map.

In [ ]:
def triage(psi_score, mae_ratio):
    """One week's monitoring verdict, from two numbers."""
    moved = psi_score >= 0.25
    worse = mae_ratio > 1.5
    if moved and worse:
        return "drift confirmed: retrain"
    if moved:
        return "early warning: watch"
    if worse:
        return "error with no input drift: investigate"
    return "healthy"


report = pd.DataFrame({
    "psi": week_psi.round(3),
    "mae_vs_launch": (weekly["mae"] / launch_mae).round(2),
})
report["verdict"] = [triage(p, r) for p, r in
                     zip(report["psi"], report["mae_vs_launch"])]
print(report.to_string())

fig, ax = plt.subplots(figsize=(7.5, 4.6))
ax.axvline(0.25, color="#6B7785", linestyle=":")
ax.axhline(1.5, color="#6B7785", linestyle=":")
for (x, y, word) in [
        (0.02, 0.04, "healthy"),
        (0.98, 0.04, "early warning"),
        (0.02, 0.96, "error, no input drift"),
        (0.98, 0.96, "drift confirmed")]:
    ax.text(x, y, word, transform=ax.transAxes, fontsize=9,
            color="#6B7785", ha="left" if x < 0.5 else "right",
            va="bottom" if y < 0.5 else "top")
dot_colours = ["#9E3B33" if v.startswith("drift") else "#2C5F79"
               for v in report["verdict"]]
ax.scatter(report["psi"], report["mae_vs_launch"], s=60,
           color=dot_colours, zorder=3)
for week, row in report.iterrows():
    ax.annotate(f"w{week}", (row["psi"], row["mae_vs_launch"]),
                xytext=(6, -3 if week % 2 else 5),
                textcoords="offset points", fontsize=8)
ax.set_xscale("log")
ax.set_xlim(0.02, 2.5)
ax.set_ylim(0.5, 3.8)
ax.set_xlabel("PSI of distance (log scale)")
ax.set_ylabel("MAE / launch MAE")
ax.set_title("The monitoring map: one dot per week", fontsize=10)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

Weeks 1 to 5 sit in **healthy**, including the week-3 false alarm.
Weeks 6, 7 and 8 sit in **drift confirmed**, and climb further up
and to the right each week. The verdict for those three weeks is
the same: retrain.

Two boxes are empty in our story. Think about what would put a week
in each one:

- **Early warning, error normal.** The monsoon arrives and half the
  orders are in the rain. The `rain` column shifts, but the model
  learned what rain does, so it stays accurate. Watch; do not
  retrain.
- **Error high, no input drift.** A new traffic rule makes every
  trip 5 minutes slower. The orders look exactly the same; only the
  answers changed. This is **pure concept drift**, and only the
  error can catch it --- which is why you cannot monitor inputs
  alone.

The map reads one week at a time. The alert rule from Step 8 adds
"two weeks running" on top of it.

### Step 11 --- The numbers a service should publish

Prometheus scrapes a `/metrics` endpoint every few seconds and
stores what it finds. You do not need Prometheus today --- you need
to know which numbers are worth publishing.

**Counters** only go up. **Gauges** go up and down. Never publish a
customer's data here; this endpoint is usually open to the whole
company.

In [ ]:
metrics = {
    "predictions_total": len(live),
    "prediction_errors_total": 3,
    "prediction_duration_seconds_avg": 0.004,
    "model_mae_minutes": float(weekly["mae"].iloc[-1]),
    "input_distance_psi": psi(
        X_train["distance_km"],
        live.loc[live["week"] == 8, "distance_km"]),
    "model_version": "1.0.0",
}

print("# what /metrics would return")
for name, value in metrics.items():
    if isinstance(value, float):
        print(f"{name} {value:.4f}")
    else:
        print(f"{name} {value}")

`model_mae_minutes` and `input_distance_psi` are the two that matter
and the two most teams forget. Uptime and latency tell you the
service is *running*. Only these tell you it is *right*.

### Step 12 --- Accurate for whom?

Now the audit. The overall MAE for week 8 is one number. Split it by
group and see whether everyone gets the same quality of service.

We look at three splits: rain, traffic, and distance.

In [ ]:
week8 = live[live["week"] == 8].copy()
week8["error"] = (week8["actual_min"] - week8["predicted_min"]).abs()

print(f"Week 8 overall MAE: {week8['error'].mean():.2f} minutes")
print()

def error_by(frame, column, labels=None):
    table = (frame.groupby(column, observed=True)["error"]
             .agg(["size", "mean"]))
    table.columns = ["orders", "mae"]
    if labels:
        table.index = [labels.get(i, i) for i in table.index]
    return table.round(2)

print("By weather:")
print(error_by(week8, "rain", {0: "dry", 1: "raining"}).to_string())
print()
print("By traffic:")
print(error_by(week8, "traffic_level",
               {1: "light", 2: "medium", 3: "heavy"}).to_string())
print()

week8["trip"] = pd.cut(week8["distance_km"],
                       bins=[0, 5, 12, 100],
                       labels=["short (<5km)", "medium (5-12km)",
                               "long (>12km)"])
print("By distance:")
print(error_by(week8, "trip").to_string())

Read the distance table. Short trips are still served well. Long
trips --- the ones that did not exist in the training data --- are
served far worse.

The average hid that completely. And the customers on the wrong end
of it are the ones furthest from the centre of town, who are
already waiting longest.

### Step 13 --- Draw the audit, with the counts on the bars

One chart per split. The height is the error; the number on each
bar is how many orders are in that group, because a terrible
score on four orders is noise and on seventy is a problem.

A group turns **red** only when it is served at least half again
as badly as the best group in its chart. Small differences stay
grey: they are the ordinary noise you saw in week 3.

In [ ]:
splits = [("rain", {0: "dry", 1: "raining"}),
          ("traffic_level", {1: "light", 2: "medium", 3: "heavy"}),
          ("trip", None)]

fig, axes = plt.subplots(1, 3, figsize=(13, 3.4), sharey=True)
for ax, (column, labels) in zip(axes, splits):
    table = error_by(week8, column, labels)
    worst, best = table["mae"].max(), table["mae"].min()
    unfair = worst > 1.5 * best          # clearly worse, not noise
    colours = ["#9E3B33" if (unfair and v == worst) else "#6B7785"
               for v in table["mae"]]
    bars = ax.bar([str(i) for i in table.index], table["mae"],
                  color=colours)
    ax.bar_label(bars, labels=[f"n={n}" for n in table["orders"]],
                 padding=3, fontsize=8)
    ax.axhline(week8["error"].mean(), color="#2C5F79",
               linestyle="--", linewidth=1)
    ax.set_title(f"by {column}", fontsize=10)
    ax.tick_params(axis="x", labelsize=8)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("MAE in week 8 (minutes)")
fig.suptitle("Dashed line: the overall average. "
             "It describes nobody in the right-hand chart.",
             fontsize=10)
plt.tight_layout()
plt.show()

The left two charts stay grey: rain and traffic are served about
equally, and the small differences are noise. The right one is
not. One red bar, five times the height of its neighbours, on the
largest group of all. That is the chart you would put in front of
the person who signs the model off.

### Step 14 --- Is the gap real, or a few bad orders?

A bar shows an average, and a handful of disasters can drag an
average up. Before you tell anyone a group is served badly, look at
**every order** in it.

Below, each dot is one week-8 order. The box covers the middle half
of each group. The dotted line at +10 marks a customer who waited
**more than 10 minutes longer than promised** --- the kind who
writes a one-star review.

The table then turns MAE into a number a manager understands: what
share of each group's customers crossed that line.

In [ ]:
groups = ["short (<5km)", "medium (5-12km)", "long (>12km)"]
jitter = np.random.default_rng(0)     # spreads the dots; not data

fig, ax = plt.subplots(figsize=(8, 4))
for i, group in enumerate(groups):
    errs = week8.loc[week8["trip"] == group, "signed_error"]
    ax.boxplot(errs, positions=[i], widths=0.5, showfliers=False,
               medianprops={"color": "#6B7785"})
    ax.scatter(i + jitter.uniform(-0.18, 0.18, len(errs)), errs,
               s=12, alpha=0.7, zorder=3,
               color="#9E3B33" if group.startswith("long")
               else "#2C5F79")
ax.axhline(0, color="#6B7785", linewidth=1)
ax.axhline(10, color="#9E3B33", linestyle=":", linewidth=1)
ax.text(0.5, 10.5, "10 min later than promised", fontsize=8,
        color="#9E3B33", ha="center")
ax.set_xticks(range(len(groups)), groups)
ax.set_ylabel("actual - predicted (minutes)")
ax.set_title("Week 8, every order", fontsize=10)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

late_share = {}
print(f"{'group':<18}{'orders':>7}{'MAE':>7}"
      f"{'avg signed':>12}{'>10 min late':>14}")
for group in groups:
    errs = week8.loc[week8["trip"] == group, "signed_error"]
    late_share[group] = float((errs > 10).mean())
    print(f"{group:<18}{len(errs):>7}{errs.abs().mean():>7.2f}"
          f"{errs.mean():>+12.2f}{late_share[group]:>14.0%}")

**How to read it.** For short and medium trips the box straddles
zero: small misses, early and late in about equal measure. That is
a healthy model.

For long trips the **whole box** sits above zero: only one dot of
74 is below it, and more than half are above the 10-minute line.
This is not three unlucky orders dragging an average up; the
entire group has moved. With 74 orders, it is not noise either.

The last column is the sentence to say out loud: *"short-trip
customers are almost never badly late; long-trip customers often
are."* Nobody outside this room will remember an MAE. They will
remember that.

### Step 15 --- The audit checklist

The last check before the repair. Before any model goes live, and
every quarter afterwards, somebody should be able to answer these.

In [ ]:
checklist = [
    ("Who is worst served?",
     "split the error by every group you can name"),
    ("How would we know it broke?",
     "name the metric, the threshold and who gets the alert"),
    ("How fast would we know?",
     "inputs are instant; errors wait for the truth to arrive"),
    ("What was it trained on?",
     "dates, row count, and which groups were thin or missing"),
    ("Can we explain one prediction?",
     "to a customer, not to a data scientist"),
    ("Can we turn it off?",
     "what serves traffic while the model is switched out"),
    ("What did we log?",
     "enough to investigate, nothing that identifies a person"),
    ("Who signed it off?",
     "a name and a date, not a team"),
]

for i, (question, how) in enumerate(checklist, 1):
    print(f"{i}. {question}")
    print(f"   -> {how}")

print()
print("Our model fails number 1 today: long trips are served")
print("far worse than short ones. The honest fix is to retrain")
print("on data that includes the new suburb -- not to report")
print("the average and hope. That retrain is Practical P14.")

### Step 16 --- Write it down: the incident note

Charts convince the people in the room. A short written note
convinces the people who were not --- the manager, the next team,
you in three months.

A good note answers five questions in plain words: **what** we
saw, **when**, **who** is affected, **why** it happened, and
**what** we will do. Every number below is read from a variable
computed earlier in this notebook, so the note cannot disagree with
the charts.

This is the shape of the markdown answers you submit.

In [ ]:
trip_table = error_by(week8, "trip")
long_row, short_row = (trip_table.loc["long (>12km)"],
                       trip_table.loc["short (<5km)"])

lines = [
    f"INCIDENT NOTE -- delivery-time model {metrics['model_version']}",
    "",
    f"WHAT    Error rose from {launch_mae:.2f} min at launch "
    f"to {weekly['mae'].iloc[-1]:.2f} min in week 8.",
    f"WHEN    Inputs shifted in week {first_shift} "
    f"(PSI {week_psi[first_shift]:.2f}); "
    f"the error alert fired in week {first_alert}.",
    f"WHO     Long trips (>12 km): MAE {long_row['mae']:.2f} min "
    f"on {int(long_row['orders'])} orders, "
    f"against {short_row['mae']:.2f} for short trips.",
    f"        {late_share['long (>12km)']:.0%} of long-trip "
    f"customers waited 10+ min longer than promised.",
    f"WHY     No training trip was longer than "
    f"{longest_trained:.1f} km. Beyond that, the model",
    f"        under-predicts by {beyond_bias:.1f} min on average; "
    f"inside it, MAE is still {inside_mae:.2f}.",
    "ACTION  Retrain on data that includes the new suburb (P14).",
    "        Until then, add a safety margin to long-trip estimates.",
]
note = "\n".join(lines)

print(note)
(WORK / "incident_note.txt").write_text(note, encoding="utf-8")
print("saved to", WORK / "incident_note.txt")

Read the note as if you had never seen this notebook. It names a
group, a size, a cause and an action, and it fits on a phone
screen. Notice what it does **not** say: "the model's MAE is 6.50".
That one number, on its own, describes nobody.

---

# Your turn

The walkthrough above is finished. Now you write some code.

There are **3 tasks**. Each one is small. Each one has a hint.
Do them in order.

Where you see `# TODO`, replace that line with your own code. Do not delete
the variable name on the left of the `=` sign --- the self-check at the
bottom looks for exactly that name.

When you have tried all three, run the **self-check** cell at the end. It
prints a table telling you which tasks are correct. You can run it as many
times as you like.

### Task T1 --- Drift in a different feature


    We watched `distance_km`. Check whether `prep_time_min` drifted too.

    Compute the PSI of `prep_time_min` for **week 8** against the training
    data, and put it in `T1_prep_psi`.

    Then put the string `"stable"`, `"watching"` or `"shifted"` in
    `T1_reading`, using the same thresholds as Step 6 (below 0.1, below
    0.25, otherwise).


> **Hint.** The `psi` function is already written. Pass `X_train["prep_time_min"]` as the reference and the week-8 values as the current sample. Select those with `live.loc[live["week"] == 8, "prep_time_min"]`.

In [ ]:
# TODO: PSI of prep_time_min in week 8 versus training
T1_prep_psi = None

# TODO: "stable", "watching" or "shifted"
T1_reading = None

print(T1_prep_psi, T1_reading)

### Task T2 --- A stricter alert


    The team decides the current rule is too slow. Apply `check_alerts`
    again with a **1.2x** threshold that must persist for **1** week, and
    store the returned list in `T2_alerts`.

    Then put the number of the **first** week that alerts in
    `T2_first_week`.


> **Hint.** `check_alerts(weekly, launch_mae, multiplier=1.2, in_a_row=1)`. The first week is `T2_alerts[0]["week"]` --- but check the list is not empty first.

In [ ]:
# TODO: run check_alerts with the stricter settings
T2_alerts = None

# TODO: the first week that fires
T2_first_week = None

print(T2_first_week, T2_alerts)

### Task T3 --- Who does this model serve worst?


    Using the `week8` table, build `T3_by_trip`: a dictionary mapping each
    trip-length label (`"short (<5km)"`, `"medium (5-12km)"`,
    `"long (>12km)"`) to that group's **mean absolute error**.

    Then put the label of the **worst-served** group in `T3_worst`, and its
    error in `T3_worst_mae`.


> **Hint.** `week8` already has both an `error` column and a `trip` column from Step 12. `week8.groupby("trip", observed=True)["error"].mean().to_dict()` gives you the dictionary in one line. The worst is `max(T3_by_trip, key=T3_by_trip.get)`.

In [ ]:
# TODO: mean error for each trip-length group, as a dictionary
T3_by_trip = None

# TODO: which group has the highest error, and what is it?
T3_worst = None
T3_worst_mae = None

print(T3_by_trip)
print("worst served:", T3_worst, "at", T3_worst_mae, "minutes")

---

## Self-check

Run the cell below to mark your work.

In [ ]:
# ------------------------------------------------------------------
# SELF-CHECK --- run this when you have attempted the tasks above.
# It never breaks your notebook. A task you have not done yet simply
# shows FAIL.
# ------------------------------------------------------------------

_results = []


def _check(label, fn):
    """Evaluate one graded condition without ever raising."""
    try:
        ok = bool(fn())
    except Exception:
        ok = False
    _results.append((label, ok))


_check('T1 | T1_prep_psi is the PSI of prep_time_min in week 8', lambda: abs(float(T1_prep_psi) - psi(X_train['prep_time_min'], live.loc[live['week'] == 8, 'prep_time_min'])) < 1e-9)
_check('T1 | the reading matches the thresholds', lambda: T1_reading == ('stable' if float(T1_prep_psi) < 0.1 else ('watching' if float(T1_prep_psi) < 0.25 else 'shifted')))
_check('T1 | prep time did not drift, unlike distance', lambda: T1_reading == 'stable')
_check('T2 | the stricter rule produces alerts', lambda: isinstance(T2_alerts, list) and len(T2_alerts) > 0)
_check('T2 | T2_alerts matches check_alerts at 1.2x for 1 week', lambda: [a['week'] for a in T2_alerts] == [a['week'] for a in check_alerts(weekly, launch_mae, multiplier=1.2, in_a_row=1)])
_check('T2 | it fires earlier than the original rule', lambda: int(T2_first_week) <= check_alerts(weekly, launch_mae)[0]['week'])
_check('T3 | all three trip-length groups are reported', lambda: set(str(k) for k in T3_by_trip) == {'short (<5km)', 'medium (5-12km)', 'long (>12km)'})
_check('T3 | T3_worst is the group with the highest error', lambda: T3_worst == max(T3_by_trip, key=T3_by_trip.get))
_check("T3 | T3_worst_mae is that group's error", lambda: abs(float(T3_worst_mae) - float(T3_by_trip[T3_worst])) < 1e-6)
_check('T3 | long trips are served worse than short ones', lambda: float(T3_by_trip['long (>12km)']) > float(T3_by_trip['short (<5km)']))

print("==================================================================")
print("SELF-CHECK   Practical 13 --- Watching a Model Get Worse")
print("==================================================================")
for _label, _ok in _results:
    print(f"  [{'PASS' if _ok else 'FAIL'}]  {_label}")
print("------------------------------------------------------------------")
_passed = sum(1 for _, _ok in _results if _ok)
print(f"  {_passed} of {len(_results)} checks passed")
print("==================================================================")
if _passed == len(_results):
    print("Well done. Save the notebook and submit it.")
else:
    print("Read the FAIL lines above, fix those tasks, run this cell again.")

---

## What to submit

1. This notebook, with every cell run and every chart visible.
2. In a markdown cell, your answer to checklist question 1 for this model: which group is worst served, by how much, and what you would do about it. Write it in your own words; the incident note from Step 16 shows the shape.
3. In a markdown cell, the alert rule you would actually configure: the metric, the threshold, how long it must persist, and who is woken up.

Name your file `PXX_<your-roll-number>.ipynb` before you upload it.

### How this practical is marked

| What is marked | Marks |
|---|---|
| Walkthrough run end to end, all charts produced | 3 |
| Task T1 --- drift measured with PSI | 2 |
| Task T2 --- an alert rule applied correctly | 2 |
| Task T3 --- the worst-served group identified | 3 |
| **Total** | **10** |

---

## Read more

- Prometheus --- Metric types --- <https://prometheus.io/docs/concepts/metric_types/>
- Prometheus --- Alerting rules --- <https://prometheus.io/docs/prometheus/latest/configuration/alerting_rules/>
- Grafana --- Introduction to dashboards --- <https://grafana.com/docs/grafana/latest/dashboards/>
- Fairlearn --- Assessing fairness --- <https://fairlearn.org/main/user_guide/assessment/index.html>
- scikit-learn --- Metrics and scoring --- <https://scikit-learn.org/stable/modules/model_evaluation.html>